# DistilBERT fine-tune — CFPB complaint classifier (section 19)

Runs on Colab with a **T4 GPU**: Runtime -> Change runtime type -> T4 GPU.

Upload `data/train.parquet`, `data/val.parquet`, `data/test.parquet` from the repo, then Run all.
Takes roughly 10-20 minutes. Start it, then go build the baseline (section 18) while it runs.

`MAX_LEN` must stay at 256: it has to match `model/export_onnx.py` and `model/metadata.json`,
or serving pads to a length the model never trained on.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn pandas pyarrow

## Upload the three splits

In [ ]:
from google.colab import files

uploaded = files.upload()   # select train.parquet, val.parquet, test.parquet
sorted(uploaded)

In [ ]:
import pandas as pd

train_df = pd.read_parquet('train.parquet')
val_df   = pd.read_parquet('val.parquet')
test_df  = pd.read_parquet('test.parquet')

LABELS = sorted(train_df.label.unique())          # same order as model/labels.json
label2id = {l: i for i, l in enumerate(LABELS)}
print(len(train_df), len(val_df), len(test_df), LABELS)

## Tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = 'distilbert-base-uncased'
MAX_LEN = 256                                     # must match export_onnx.py

tok = AutoTokenizer.from_pretrained(MODEL)

def prepare(df):
    ds = Dataset.from_pandas(df[['text', 'label']].assign(labels=df.label.map(label2id)), preserve_index=False)
    ds = ds.map(lambda b: tok(b['text'], truncation=True, padding='max_length', max_length=MAX_LEN), batched=True)
    return ds.remove_columns(['text', 'label'])

train_ds, val_ds, test_ds = prepare(train_df), prepare(val_df), prepare(test_df)

## Train

2 epochs (DistilBERT overfits 8k rows after that), lr 2e-5, fp16 mixed-precision training
— unrelated to the INT8 *inference* quantization in section 10. The best checkpoint is
selected on **validation** macro-F1; the test split stays untouched until the cell after.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, Trainer, TrainingArguments)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=len(LABELS),
    id2label={i: l for i, l in enumerate(LABELS)},
    label2id=label2id,
)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        'accuracy': accuracy_score(p.label_ids, preds),
        'macro_f1': f1_score(p.label_ids, preds, average='macro'),
        'micro_f1': f1_score(p.label_ids, preds, average='micro'),
    }

args = TrainingArguments(
    output_dir='out',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=True,
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, compute_metrics=metrics)
trainer.train()

## Test-set results — copy these into `docs/measurements.md`

In [ ]:
import json
from sklearn.metrics import classification_report

pred = trainer.predict(test_ds)
print(pred.metrics)

y_pred = [LABELS[i] for i in pred.predictions.argmax(1)]
y_true = [LABELS[i] for i in pred.label_ids]
print(classification_report(y_true, y_pred, digits=4))

json.dump(classification_report(y_true, y_pred, digits=4, output_dict=True),
          open('distilbert_metrics.json', 'w'), indent=2)

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=LABELS,
                                        xticks_rotation=45, colorbar=False, cmap='Blues', ax=ax)
fig.tight_layout()
fig.savefig('distilbert_confusion_matrix.png', dpi=150)

## Save and download

Unzip into the repo root as `distilbert-complaints/`, then section 20:

```bash
python model/export_onnx.py --src ./distilbert-complaints --labels model/labels.json
python model/quantize.py
docker build --platform linux/amd64 -f docker/inference/Dockerfile -t complaint-inference:v2-int8 .
```

In [ ]:
trainer.save_model('distilbert-complaints')
tok.save_pretrained('distilbert-complaints')

!zip -qr distilbert-complaints.zip distilbert-complaints distilbert_metrics.json distilbert_confusion_matrix.png
files.download('distilbert-complaints.zip')